# Propiedades

Notebook para la implementación indiviual de propiedades para su posterior integración con el resto del código.

In [6]:
import networkx as nx
import numpy as np
from freyrelab.nets.paths2 import ShortestPaths, Efficiency, ShortestDistances
from freyrelab.regnets.regnet import RegNet


### "boilerplate code", no modificar ⚠️

In [2]:
from abc import ABC, abstractmethod

# Decorators for graph characteristics

def use_direction(cls):
    cls._use_direction = True
    return cls

def use_selfloops(cls):
    cls._use_selfloops = True
    return cls

def use_giant_component(cls):
    cls._use_giant_component = True
    return cls

def return_scalar(cls):
    cls._return_type = "scalar"
    return cls

def return_distribution(cls):
    cls._return_type = "distribution"
    return cls

def use_paths(cls):
    cls._use_paths = True
    return cls

class _Property(ABC):
    """Abstract base class for all properties."""

    _return_type = None
    _use_paths = False
    _use_direction = False
    _use_selfloops = False
    _use_giant_component = False

    def __init__(self, G: nx.DiGraph):
        self.G = G
        self._raw_value = None
        self._n_nodes = self.G.number_of_nodes()
        if self._n_nodes == 0:
            raise NullGraphError("A null graph has no self-regulations.")

    @abstractmethod
    def compute(self):
        return self._raw_value

    @abstractmethod
    def norm_biol(self, *args):
        pass

    @abstractmethod
    def norm_network(self, *args):
        pass




In [3]:
class NormalizationError(Exception):
    """Exception raised for errors in the normalization.

    Attributes:
        message -- explanation of the error
    """

    def __init__(self, message="An error occurred during normalization."):
        self.message = message
        super().__init__(self.message)

class NullGraphError(Exception):
    """Exception raised for null graph."""
    pass

class EmptyGraphError(Exception):
    """Exception raised for empty graph. Nodes with no edges."""

    pass

def check_raw_value(func):
    """Decorator to check if raw value is None. If it is, raise an error."""
    def wrapper(self, *args):
        if self._raw_value is not None:
            return func(self, *args)
        else:
            raise ValueError("Raw value is None. Call compute() method first.")
    return wrapper


### Describimos la propiedad particular:
Incluye al menos una referencia de su uso previo, de preferencia en redes biológicas.
Para incluir citas, usa el PubMed ID (PMID) de la publicación entre corchetes. Para múltiples citas, asigna sólo una detro de cada par de corchetes, sin espacios entre citas: [PMID1][PMID2][PMID3]

    TU DEFINICIÓN AQUÍ 👇

The global efficiency of a graph is the average of the inverse of the shortest paths between all pairs of nodes. It was introdued by Latora and Marchiori to meassure information exhange efficiency [11690461] and solves the divergence problem of network average dsitance due to disconnected graphs [10.1080/00018730601170527]. Dense networks will have higher efficiency than sparse networks because of shorter shortest paths (shortcuts).

### Definimos la clase hija y sus métodos o funciones auxiliares:

    YOUR CODE HERE 👇

In [5]:
# Esta función auxiliar no se incluyó como método de la clase Density porque se utilizará
# cada vez que se requiera la lista de TFs
def get_parent_nodes(G: nx.DiGraph):
    """Get the parent nodes of a graph."""
    return [i for i, k_out in G.out_degree() if k_out > 0]

# Auxiliary functions
def remove_self_loops(G: nx.DiGraph):
    G.remove_edges_from(nx.selfloop_edges(G))
    return G


@use_paths
@return_distribution
class GlobalEfficiency(_Property):
    """Global efficiency of a graph.

    The global efficiency of a graph is the average of the inverse of the shortest paths between all pairs of nodes.
    """
    __name__ = 'Global Efficiency'

    def __init__(self, G: nx.DiGraph):
        """
        Args:
            G (nx.DiGraph): Graph.
        """
        G = remove_self_loops(G)
        if not G.number_of_edges():
            raise EmptyGraphError("Graph has no edges.")
        G = G.to_undirected()
        super().__init__(G)
    
    def compute(self) -> np.array:
        """Compute the betweenness centrality for each node in the graph.

        Returns:
            np.array: Betweenness centrality for each node in the graph.
        """
        shortest_distances_G = ShortestDistances(self.G)
        efficiency = Efficiency(self.G, shortest_distances_G)
        self._raw_value = efficiency.global_efficiency
        return self._raw_value
    
    @check_raw_value
    def norm_biol(self) -> None:
        raise NormalizationError("Betweenness centrality cannot be normalized by biological properties.")
    
    @check_raw_value
    def norm_network(self) -> np.array:
        """Normalize betweenness centrality by the combinatory number of pairs of nodes excluding the node itself.

        Returns:
            np.array: Normalized betweenness centrality for each node in the graph.
        """
        # Already normalized
        return self._raw_value

### Pruebas de tu código
Agrega evaluaciones a tu código con redes que se conoce su valor teórico. Un grafo vacío y uno completo es de los casos más comunes, pero puedes agregar los que consideres necesario.

    TUS PRUEBAS AQUÍ 👇

In [19]:
# Para densidad usaremos un grafo completo dirigido con self loops y uno sin ninguna interacción, sólo nodos.
import pytest
import networkx as nx


# null graph
G = RegNet(nx.DiGraph())
with pytest.raises(EmptyGraphError) as e_info:
    property = GlobalEfficiency(G)

# empty graph with nodes
G = RegNet(nx.DiGraph())
n_nodes = 10
G.add_nodes_from(range(n_nodes))    # add nodes, no edges

with pytest.raises(EmptyGraphError) as e_info:
    property = GlobalEfficiency(G)


# add edges
# complete directed graph with self loops (self loops are removed in the constructor)
G.add_edges_from([(i, j) for i in range(n_nodes) for j in range(n_nodes)])
property = GlobalEfficiency(G)
assert property.compute() == 1 # all nodes are connected, super efficient!
with pytest.raises(NormalizationError) as e_info:
    assert property.norm_biol()
assert property.norm_network() == 1 # all nodes are connected, super efficient!


# only half of the nodes are parents and regulate every node in the graph
import networkx as nx
G = RegNet(nx.DiGraph())
n_nodes = 10
G.add_nodes_from(range(n_nodes))
G.add_edges_from([(i, j) for i in range(n_nodes//2) for j in range(n_nodes)])
property = GlobalEfficiency(G)
G_und = G.to_undirected()
assert property.compute() == pytest.approx(0.8888, rel=1e-3)
assert property.norm_network() == pytest.approx(0.8888, rel=1e-3)


In [16]:
property.compute()

0.8888888888888888